In [11]:
import os
import sys
import time
import intake
from tqdm import tqdm
import xarray as xr
import pandas as pd
import numpy as np
import pickle
from datetime import datetime
import metpy.calc as mpcalc
from metpy.units import units


from dask.distributed import Client, LocalCluster

os.environ['WXSYSLIBDIR']='/g/data/gb02/mb0427/WxSysLib'
sys.path.append(os.environ['WXSYSLIBDIR'])
os.environ['SWTLIBDIR']='/home/565/mb0427/gdata-gb02/Australian_synoptic_weather_types'
sys.path.append(os.environ['SWTLIBDIR'])

from assign.assign import assign_grid_to_SWT

In [2]:
from utils.general.nci_utils import get_GADI_ERA5_filename
from utils.general.date_utils import generate_datetimes,generate_datetimes_months

In [3]:
def process_single_timestep(ua_data, va_data):
    """
    Processes a single time step and returns the result as a pandas Series.
    This function will be called on a single chunk of data by a Dask worker.
    """
    swt_result = assign_grid_to_SWT(
        ua_data,
        va_data,
        latname="lat",
        lonname="lon",
        #cluster_filename=f"{os.environ['SWTLIBDIR']}/SWT_fields/SWT_data.nc",
        cluster_filename=f"/g/data/if69/mb0427/SWTs/ERA5/SWT_data_reassigned_850.nc",
        interpolation=True,
        silence_warning=True,
        quiet=True
    )
    
    return pd.Series({'SWT': swt_result, 'time': pd.to_datetime(ua_data.time.item())})

In [4]:
udir=f"/g/data/ux62/access-s2/hindcast/raw_model/atmos/uas/daily/e01"
ufn=sorted(os.listdir(udir))
vdir=f"/g/data/ux62/access-s2/hindcast/raw_model/atmos/vas/daily/e01"
vfn=sorted(os.listdir(vdir))
dt_list=[f.split('da_uas_')[1].split(f'_e01.nc')[0] for f in ufn]

In [13]:
wind_speed = mpcalc.wind_speed(u*units('m/s'), v*units('m/s'))

ValueError: operands could not be broadcast together with shapes (279,324,432) (279,325,432) 

In [5]:
dt_list=[dt for dt in dt_list if int(dt[0:6])==198101]

In [14]:
n=1
indt=dt_list[0]
member=f"e{n:0{2}d}"

datapath=f"/g/data/if69/mb0427/SWTs/Seasonal/access-s2"

udir=f"/g/data/ux62/access-s2/hindcast/raw_model/atmos/uas/daily/{member}"
vdir=f"/g/data/ux62/access-s2/hindcast/raw_model/atmos/vas/daily/{member}"

ufn=f'{udir}/da_uas_{indt}_{member}.nc'
vfn=f'{vdir}/da_vas_{indt}_{member}.nc'
u=xr.open_dataset(ufn)['uas']
v=xr.open_dataset(vfn)['vas']

new_lat = np.arange(-90, 90.25, 0.25)
new_lon = np.arange(0, 360, 0.25)   # or -180 to 180 if that matches your data

In [ ]:
for t in tqdm(u.time):
    wndspd=np.hypot(u.sel(time=t),v.sel(time=t))

  0%|          | 0/279 [00:00<?, ?it/s]

In [ ]:
u_i = u.rename({'lon_1':'lon'}).interp(lat=new_lat, lon=new_lon)
v_i = v.rename({'lon_1':'lon'}).interp(lat=new_lat, lon=new_lon)

In [ ]:
for indt in tqdm(dt_list[0:1]):
    df=[]
    for n in range(1,10):  
        try:
            member=f"e{n:0{2}d}"
            
            datapath=f"/g/data/if69/mb0427/SWTs/Seasonal/access-s2"
    
            udir=f"/g/data/ux62/access-s2/hindcast/raw_model/atmos/uas/daily/{member}"
            vdir=f"/g/data/ux62/access-s2/hindcast/raw_model/atmos/vas/daily/{member}"
            
            ufn=f'{udir}/da_uas_{indt}_{member}.nc'
            vfn=f'{vdir}/da_vas_{indt}_{member}.nc'
            u=xr.open_dataset(ufn)['uas']
            v=xr.open_dataset(vfn)['vas']
            wndspd=np.hypot(u,v)
            

            
        
            #results=[]
            #for t in u.time:
            #    results.append(process_single_timestep(u.sel(time=t), v.sel(time=t)))
            #    #print(process_single_timestep(u.sel(time=t), v.sel(time=t)))
            #temp = (
            #pd.concat(results, axis=1)
            #  .T
            #  .infer_objects()          # <-- key line
            #  .set_index('time')
            #  .reset_index()
            #)
    
            #temp['number'] = member
            #df.append(temp)
        except Exception as e:
            print(f"An error occurred: {e}")
    
    #df=pd.concat(df)
    #df['init'] = datetime.strptime(indt,"%Y%m%d")
    #outfile=f"{datapath}/access-s2_hindcast_{indt}.pkl" 
    #df.to_pickle(outfile)

In [4]:
datapath=f"/g/data/if69/mb0427/SWTs/Seasonal/access-s2"
datapath=f"/g/data/if69/mb0427/SWTs/Seasonal/access-s2"
infile=f"{datapath}/access-s2_hindcast_19810211.pkl"
pd.read_pickle(infile)#.SWT.unique()

,time,SWT,number,init
0,1981-02-12,AM-D,e01,1981-02-11
1,1981-02-13,AM-D,e01,1981-02-11
2,1981-02-14,AM-D,e01,1981-02-11
3,1981-02-15,AM-E,e01,1981-02-11
4,1981-02-16,AM-C,e01,1981-02-11
...,...,...,...,...
274,1981-11-13,EH-E,e03,1981-02-11
275,1981-11-14,EH-E,e03,1981-02-11
276,1981-11-15,EH-E,e03,1981-02-11
277,1981-11-16,TH-B,e03,1981-02-11
